# Import

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("/exp/sbnd/data/users/lnguyen/cafpyana_pi0")

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import warnings

import cafpybara.analyses.hnlpi0 as ana

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

<h1>Selection Mode</h1>

In [ ]:
selection_mode = '1shw'

cuts_for_load = ana.PI0_CUT_LISTS[selection_mode]

<h1>Load Files</h1>

In [ ]:
FILEPATH = '/exp/sbnd/data/users/lnguyen/cafpyana_pi0/dataframes/July2026/'

DTBNB_FILES = [
    FILEPATH + "dtbnb_fix.df",
]
DTOFF_FILE = FILEPATH + "dtoff.df"

MCBNB_FILE = FILEPATH + "mcbnb_cv.df"

MCHNL_FILES = {
    140: FILEPATH + "mchnl_nupi0_m140.df",
    165: FILEPATH + "mchnl_nupi0_m165.df",
    190: FILEPATH + "mchnl_nupi0_m190.df",
    215: FILEPATH + "mchnl_nupi0_m215.df",
    240: FILEPATH + "mchnl_nupi0_m240.df",
    260: FILEPATH + "mchnl_nupi0_m260.df",
}
m_list = [260]

rec_tree = "rec"

<h3> Data BNB </h3>

In [ ]:
dtbnb_pot = 0
dtbnb_ngates = 0
dtbnb_df = None

for f in DTBNB_FILES:
    thisdf, this_pot, this_ngates = ana.load_data(
        f, onbeam=True, rec_key=rec_tree, cuts=cuts_for_load,
        preprocess_fn=ana.preprocess_databnb,
    )
    dtbnb_pot += this_pot
    dtbnb_ngates += this_ngates

    if dtbnb_df is None:
        dtbnb_df = thisdf
    else:
        dtbnb_df = pd.concat([dtbnb_df, thisdf], ignore_index=True)

print("Data POT: ", dtbnb_pot)


<h3>MC BNB</h3>

In [ ]:
mcbnb_df, mcbnb_pot, mcbnb_ngen = ana.load_mc(
    MCBNB_FILE,
    rec_key=rec_tree,
    cuts=cuts_for_load,
    preprocess_fn=ana.preprocess_mcbnb,
    define_signal_fn=lambda df: ana.define_signal_pi0(df, prefix=('slc', 'truth')),
    chunk_splits=1,
)

scale_mcbnb = dtbnb_pot/mcbnb_pot

print("mc pot: ", mcbnb_pot)
print("mc ngen: ", mcbnb_ngen)
print("scale: ", scale_mcbnb)


<h3>Data Offbeam+Light</h3>

In [ ]:
dtoff_df, _, noffbeamgates = ana.load_data(
    DTOFF_FILE, onbeam=False, rec_key=rec_tree, cuts=cuts_for_load,
    preprocess_fn=ana.preprocess_dataoff,
    offbeam_signal_value=ana.signal_dict_hnl['offbeam'],
)

f = 0.0725

ongates_per_pot = dtbnb_ngates / dtbnb_pot
noffbeamscale_mc = ((1- f)*(ongates_per_pot*mcbnb_pot))/noffbeamgates


<h3>MC MeVPrtl HNL</h3>

In [ ]:
hnl_dfs_dict = {}
hnl_extra_info_dict = {}

for mass in m_list:
    print(f"Loading HNL mass {mass} MeV...")
    print(MCHNL_FILES[mass])

    mchnl_df, mchnl_pot, mchnl_info = ana.load_mchnl(MCHNL_FILES[mass], rec_key=rec_tree, cuts=cuts_for_load)

    hnl_dfs_dict[mass] = mchnl_df
    hnl_extra_info_dict[mass] = {
        'scale_mchnl': dtbnb_pot / mchnl_pot,
        'simU': mchnl_info['simU'],
        'hnlM': mchnl_info['hnlM'],
        'mchnl_pot': mchnl_pot,
    }


<h1> Add weights_mc/systematics/sample columns </h1>

In [ ]:
mcbnb_df[('weights_mc', '', '', '', '', '')] = 1.0
dtoff_df[('weights_mc', '', '', '', '', '')] = noffbeamscale_mc

norm_col = ('flux_pot_norm', '', '', '', '', '')
mcbnb_df[norm_col] = mcbnb_df[('weights_mc', '', '', '', '', '')] / (ana.integrated_flux * (mcbnb_pot / 1e6))
dtoff_df[norm_col] = dtoff_df[('weights_mc', '', '', '', '', '')] / (ana.integrated_flux * (mcbnb_pot / 1e6))

for mass in m_list:
    hnl_dfs_dict[mass][norm_col] = hnl_dfs_dict[mass][('weights_mc', '', '', '', '', '')] / (ana.integrated_flux * (hnl_extra_info_dict[mass]['mchnl_pot'] / 1e6))

<h1>Merge Data Offbeam+Light and MC BNB, Add MC Stats</h1>

In [ ]:
mcstat_cols = ['__ntuple', 'entry', 'rec.slc..index', 'run', 'subrun', 'evt', 'sample', 'file_idx']

mcbnb_df = pd.concat([
    ana.mcstat(mcbnb_df.assign(sample=0).set_index("sample", append=True).reset_index(), cols=mcstat_cols),
    dtoff_df.assign(sample=1).set_index("sample", append=True).reset_index(),
])

for mass in m_list:
    hnl_dfs_dict[mass] = ana.mcstat(
        hnl_dfs_dict[mass].assign(sample=0).set_index("sample", append=True).reset_index(),
        cols=mcstat_cols,
    )


<h1> Systematics </h1>

In [ ]:
#define the variables to plot and the bins for histograms
var = ('slc', 'barycenterFM', 'flashTime_calib_mod')
bins = np.linspace(0, 19, 20)

In [ ]:
#load detvar samples
hnl_detvar_dict = ana.load_detvar_dict(ana.config.HNL_DETVAR_DIR + "/detvars.h5")

#mcbnb
mc_syst_output = ana.get_total_cov(
    mcbnb_df,
    reco_var=var,
    bins=bins,
    # cuts_for_load must match the selection already applied to mcbnb_df
    # above -- see README's "Common mistakes" section for what happens if
    # this is left out.
    **ana.systs_input(
        mcbnb_pot,
        cuts_for_load,
        uncertainty_keys={'rate', 'norm', 'detv'},
        event_type='all',
        detvar_dict=hnl_detvar_dict,
    ).to_kwargs(),
)

#mchnl
signal_output_dict = {}

for mass in m_list:
    hnl_syst_output = ana.get_total_cov(
        hnl_dfs_dict[mass],
        reco_var=var,
        bins=bins,
        **ana.systs_input(
            hnl_extra_info_dict[mass]['mchnl_pot'],
            cuts_for_load,
            uncertainty_keys={'rate', 'norm'},
            event_type='all',
            detvar_dict=hnl_detvar_dict,
        ).to_kwargs(),
    )
    signal_output_dict[mass] = hnl_syst_output

<h1> Plot </h1>

<h3>MC BNB + DATA BNB</h>

In [ ]:
_,ax_main, ax_sub, mcbnb_dict = ana.plot_mc_data(
                          mc_df  = mcbnb_df,
                          data_df = dtbnb_df,
                          var=var,
                          bins=bins,
                          categories=ana.background_categories_hnl,
                          ylabel=f"POT Normalized [{dtbnb_pot:.2e} POT]",
                          xlabel='Neutrino Arrival Time [ns]',
                          percents=True,
                          systs = mc_syst_output,
                          scale = scale_mcbnb,
                          bin_labels = None,
                          legend_kwargs={'loc': 'upper right', 'ncol': 1},
                          cut_val = [],
                          title = None,
                          )
ax_main.set_xlim(0,30)
ax_sub.set_xlim(0,30)

<h3>MC BNB + DATA BNB + MC HNL</h3>

In [ ]:
result_bnb_dict = {}
result_hnl_dict = {}

signal_plots_cfg = ana.PlottingConfig(
    ylabel=f"POT Normalized [{dtbnb_pot:.2e} POT]",
    xlabel="Flash Time%Period [ns]",
    categories=ana.background_categories_hnl,
    systs=mc_syst_output,
    counts=True,
    legend_kwargs={'loc': 'upper right', 'ncol': 1, 'fontsize': 8.5},
)

for mass in m_list:

    scale_hnl_display, saveU, hnl_categories = ana.hnl_categories_for_mass(mass, hnl_extra_info_dict[mass]['simU'])
    scale_hnl_plot = hnl_extra_info_dict[mass]['scale_mchnl'] * scale_hnl_display
    print(scale_hnl_display, scale_hnl_plot, hnl_extra_info_dict[260]['scale_mchnl'])

    fig, ax_main, ax_sub, mc_dict, hnl_dict, dt_dict = ana.plot_mc_hnl_data(
        mc_df=mcbnb_df,
        hnl_df=hnl_dfs_dict[mass],
        data_df=dtbnb_df,
        var=var,
        bins=bins,
        scale_nu=scale_mcbnb,
        scale_hnl=scale_hnl_plot,
        hnl_categories=hnl_categories,
        hnl_systs=signal_output_dict[mass],
        config=signal_plots_cfg,
        cut_val=[],
        title=None,
        )
    ax_main.set_xlim(0, 35)
    ax_sub.set_xlim(0, 35)

    result_bnb_dict = mc_dict
    result_hnl_dict[mass] = {}
    result_hnl_dict[mass]['saveU'] = saveU
    result_hnl_dict[mass]['result'] = hnl_dict

<h1>Systematics Breakdown</h1>

In [ ]:
fig_mc, axes_mc, cats_mc, cat_sums_mc = ana.plot_syst_category_breakdown(
    syst_vars=[(mc_syst_output, bins, "Flash Time%Period [ns] (MC BNB)")],
    category_dict=ana.category_dict_signal,
    region_label=""
)
plt.ylim(0,100)
plt.title("MC BNB Systematic Category Breakdown")
plt.show()

for mass in m_list:
    fig_hnl, axes_hnl, cats_hnl, cat_sums_hnl = ana.plot_syst_category_breakdown(
        syst_vars=[(signal_output_dict[mass], bins, f"Flash Time%Period [ns] (MC HNL {mass} MeV)")],
        category_dict=ana.category_dict_signal,
        region_label="",
    )
    plt.title(f"MC HNL {mass} MeV Systematic Category Breakdown")
    plt.ylim(0,100)
    plt.show()


<h3>Per-source drill-down</h3>

In [ ]:
cat_list = ['Flux', 'GENIE', 'DetVar']
for c in cat_list:
    if c == 'DetVar':
        fig_mc_detv, axes_mc_detv = ana.plot_syst_breakdown(
            syst_vars=[(mc_syst_output, bins, "Flash Time%Period [ns] (MC BNB)")],
            category=c,
            category_dict=ana.category_dict_signal,
        )
        plt.show()
    else:
        fig_mc, axes_mc = ana.plot_syst_breakdown(
            syst_vars=[(mc_syst_output, bins, "Flash Time%Period [ns] (MC BNB)")],
            category=c,
        category_dict=ana.category_dict_signal,
        )
        plt.show()

for mass in m_list:
    fig_hnl, axes_hnl = ana.plot_syst_breakdown(
        syst_vars=[(signal_output_dict[mass], bins, f"Flash Time%Period [ns] (MC HNL {mass} MeV)")],
        category='Flux',
        category_dict=ana.category_dict_signal,
    )
    plt.show()